# DYRK1B virtual screening - complete pipeline (Colab)
ADD workshop Olomouc 2026. Run cells 1-6 in order. Every step saves to disk; re-running continues.

1. Install + data  2. ML ranking  3. Receptor/box/hinge + docking pool  4. Docking  5. Blend + submission  6. Check

Output to submit: `submission_final.txt`

In [ ]:
# ===== 1. INSTALL + DATA =====
!pip -q install rdkit scikit-learn pandas numpy scipy
!apt-get -qq install -y openbabel > /dev/null
!wget -q -nc https://files.rcsb.org/download/8C2Z.pdb
!wget -q -nc -O vina https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64
!chmod +x vina
!git clone -q https://github.com/avnikonenko/Drug-Design-workshop-challenge-2026 2>/dev/null

import os
print("vina:", os.popen("./vina --version 2>&1").read().strip())
print("8C2Z:", os.path.exists("8C2Z.pdb"), "| obabel:", os.system("which obabel > /dev/null") == 0)

In [ ]:
# ===== 2. ML RANKING (skips if ranked_blind_set.csv exists) =====
import os, numpy as np, pandas as pd
from scipy.stats import rankdata
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import rdFingerprintGenerator as rfg
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
RDLogger.DisableLog("rdApp.*")

if os.path.exists("ranked_blind_set.csv"):
    print("ranked_blind_set.csv exists, skipping")
else:
    D = "Drug-Design-workshop-challenge-2026/data/"
    tr = pd.read_csv(D + "chembl_stereo_classes_fixed.smi", sep="\t", header=None, names=["smiles", "id", "act"])
    bl = pd.read_csv(D + "blind_stereo.smi", sep="\t", header=None, names=["smiles", "id"])
    tr["y"] = tr.act.astype(str).str.lower().isin(["1", "1.0", "active"]).astype(int)
    print("train:", len(tr), "actives:", tr.y.sum(), "| blind:", len(bl))

    morgan = rfg.GetMorganGenerator(radius=2, fpSize=2048)
    fmorgan = rfg.GetMorganGenerator(radius=2, fpSize=2048,
                                     atomInvariantsGenerator=rfg.GetMorganFeatureAtomInvGen())
    apair = rfg.GetAtomPairGenerator(fpSize=2048)

    def feats(smiles):
        bits, X, keep = [], [], []
        for i, s in enumerate(smiles):
            m = Chem.MolFromSmiles(s)
            if m is None: continue
            bits.append(morgan.GetFingerprint(m)); keep.append(i)
            X.append(np.concatenate([morgan.GetCountFingerprintAsNumPy(m),
                                     fmorgan.GetCountFingerprintAsNumPy(m),
                                     apair.GetCountFingerprintAsNumPy(m)]).astype(np.float32))
        return bits, np.vstack(X), keep

    tb, Xt, kt = feats(tr.smiles); y = tr.y.values[kt]
    bb, Xb, kb = feats(bl.smiles)
    bl = bl.iloc[kb].reset_index(drop=True)

    models = [RandomForestClassifier(n_estimators=600, class_weight="balanced_subsample", n_jobs=-1, random_state=0).fit(Xt, y),
              ExtraTreesClassifier(n_estimators=600, class_weight="balanced_subsample", n_jobs=-1, random_state=0).fit(Xt, y)]
    ml = np.mean([m.predict_proba(Xb)[:, 1] for m in models], axis=0)

    act = [b for b, l in zip(tb, y) if l == 1]
    inact = [b for b, l in zip(tb, y) if l == 0]
    def sim(fp, k=3):
        a = np.sort(DataStructs.BulkTanimotoSimilarity(fp, act))[::-1][:k].mean()
        i = np.sort(DataStructs.BulkTanimotoSimilarity(fp, inact))[::-1][:k].mean()
        return a - 0.3 * i
    sm = np.array([sim(f) for f in bb])

    bl["score"] = (rankdata(ml) + rankdata(sm)) / (2 * len(bl))
    bl.sort_values("score", ascending=False).reset_index(drop=True).to_csv("ranked_blind_set.csv", index=False)
    print("saved ranked_blind_set.csv:", len(bl))

In [ ]:
# ===== 3. RECEPTOR, DOCKING BOX, HINGE ATOMS, DOCKING POOL =====
import numpy as np, pandas as pd, subprocess
from collections import defaultdict
from rdkit import Chem

lines = [l for l in open("8C2Z.pdb") if l.startswith(("ATOM", "HETATM"))]
xyz = lambda l: np.array([float(l[30:38]), float(l[38:46]), float(l[46:54])])

prot = [l for l in lines if l.startswith("ATOM") and l[21] == "A" and l[16] in " A"]
open("rec_clean.pdb", "w").write("".join(prot) + "END\n")
subprocess.run("obabel rec_clean.pdb -O rec.pdbqt -xr -p 7.4", shell=True, check=True)

skip = {"HOH","DMS","GOL","EDO","SO4","PO4","CL","NA","MG","ZN","K","CA","PEG","ACT"}
groups = defaultdict(list)
for l in lines:
    if l.startswith("HETATM") and l[17:20].strip() not in skip:
        groups[(l[17:20], l[21], l[22:26])].append(l)
key = max(groups, key=lambda k: len(groups[k]))
lig = groups[key]
lc = np.array([xyz(l) for l in lig])
center = lc.mean(0)
size = np.minimum(np.maximum(lc.max(0) - lc.min(0) + 12, 20), 22)   # smaller box = faster

lig_no = np.array([xyz(l) for l in lig if l[76:78].strip() in ("N", "O")])
hinge = np.array([xyz(l) for l in prot if l[12:16].strip() in ("N", "O")
                  and np.min(np.linalg.norm(lig_no - xyz(l), axis=1)) < 3.4])
print("reference ligand:", key, len(lig), "atoms | center", center.round(1), "box", size.round(0))
print("hinge atoms found:", len(hinge), "(need >= 2)")

# docking pool: top 600 non-azaindoles + best 40 azaindoles
r = pd.read_csv("ranked_blind_set.csv")
azq = Chem.MolFromSmarts("c1cnc2[nH]ccc2c1")
r["az"] = [Chem.MolFromSmiles(s).HasSubstructMatch(azq) for s in r.smiles]
pool = pd.concat([r[~r.az].head(600), r[r.az].head(40)]).sort_values("score", ascending=False)
pool[["smiles", "id", "score"]].to_csv("dock_pool.csv", index=False)
print("pool:", len(pool), "| azaindoles:", int(pool.az.sum()))

In [ ]:
# ===== 4. DOCKING (stops itself at TIME_LIMIT; re-run to continue) =====
import os, time, subprocess, numpy as np, pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, rdFingerprintGenerator as rfg
from rdkit.ML.Cluster import Butina

N_DOCK     = 150        # how many compounds to dock in total
EXHAUST    = 1          # 1 = fast, 4 = more accurate but ~4x slower
TIME_LIMIT = 15 * 60    # seconds per run of this cell

pool = pd.read_csv("dock_pool.csv").reset_index(drop=True)
gen = rfg.GetMorganGenerator(radius=2, fpSize=2048)
fps = [gen.GetFingerprint(Chem.MolFromSmiles(s)) for s in pool.smiles]
d = []
for i in range(1, len(fps)):
    d += [1 - x for x in DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])]
cl = Butina.ClusterData(d, len(fps), 0.65, isDistData=True)
reps = sorted(min(c) for c in cl)                       # best member of each cluster first
order = reps + [i for i in range(len(pool)) if i not in set(reps)]
todo = pool.iloc[order[:N_DOCK]]

def prep(smi):
    m = Chem.AddHs(Chem.MolFromSmiles(smi))
    if AllChem.EmbedMolecule(m, randomSeed=7) != 0: return False
    try: AllChem.MMFFOptimizeMolecule(m, maxIters=200)
    except Exception: pass
    Chem.MolToMolFile(m, "lig.mol")
    if os.path.exists("lig.pdbqt"): os.remove("lig.pdbqt")
    subprocess.run("obabel lig.mol -O lig.pdbqt", shell=True, capture_output=True)
    return os.path.exists("lig.pdbqt") and os.path.getsize("lig.pdbqt") > 0

def dock():
    if os.path.exists("out.pdbqt"): os.remove("out.pdbqt")
    cmd = ["./vina", "--receptor", "rec.pdbqt", "--ligand", "lig.pdbqt",
           "--center_x", str(center[0]), "--center_y", str(center[1]), "--center_z", str(center[2]),
           "--size_x", str(size[0]), "--size_y", str(size[1]), "--size_z", str(size[2]),
           "--exhaustiveness", str(EXHAUST), "--num_modes", "3", "--cpu", "2",
           "--seed", "7", "--out", "out.pdbqt"]
    subprocess.run(cmd, capture_output=True, timeout=600)
    if not os.path.exists("out.pdbqt"): return None
    txt = open("out.pdbqt").read()
    res = [l for l in txt.splitlines() if l.startswith("REMARK VINA RESULT")]
    return (float(res[0].split()[3]), txt) if res else None

def hinge_hit(poses):
    for model in poses.split("ENDMDL"):
        pts = [xyz(l) for l in model.splitlines()
               if l.startswith(("ATOM", "HETATM")) and l.split()[-1] in ("N","NA","NS","OA","OS")]
        if pts and len(hinge) and np.linalg.norm(hinge[:, None] - np.array(pts)[None], axis=2).min() < 3.5:
            return 1
    return 0

done = pd.read_csv("dock_results.csv") if os.path.exists("dock_results.csv") else pd.DataFrame(columns=["id","aff","hinge"])
rows, seen = done.to_dict("records"), set(done.id)
print("already docked:", len(seen))

t0 = time.time()
for k, r_ in enumerate(todo.itertuples()):
    if r_.id in seen: continue
    if time.time() - t0 > TIME_LIMIT:
        print("time limit reached - re-run this cell to continue"); break
    try:
        if not prep(r_.smiles): continue
        out = dock()
        if out is None: continue
        rows.append({"id": r_.id, "aff": out[0], "hinge": hinge_hit(out[1])})
    except Exception as e:
        print(r_.id, "failed:", str(e)[:60])
    if k % 5 == 0:
        pd.DataFrame(rows).to_csv("dock_results.csv", index=False)
        print(f"{k+1}/{len(todo)}  {time.time()-t0:.0f}s  docked so far: {len(rows)}")
pd.DataFrame(rows).to_csv("dock_results.csv", index=False)
print("docked total:", len(rows))

In [ ]:
# ===== 5. BLEND ML + DOCKING -> submission_final.txt =====
from google.colab import files
import pandas as pd
from rdkit import Chem

W_ML, W_DOCK, W_HINGE, MAX_AZ = 0.4, 0.6, 0.3, 10

azq = Chem.MolFromSmarts("c1cnc2[nH]ccc2c1")
dk = pd.read_csv("dock_results.csv")
ml = pd.read_csv("ranked_blind_set.csv")[["id", "smiles", "score"]]
m = dk.merge(ml, on="id")
m["az"] = [Chem.MolFromSmiles(s).HasSubstructMatch(azq) for s in m.smiles]
m["final"] = (W_ML * m.score.rank(pct=True) + W_DOCK * (-m.aff).rank(pct=True) + W_HINGE * m.hinge)
m = m.sort_values("final", ascending=False)

ids, n_az = [], 0
for row in m.itertuples():
    if row.az and n_az >= MAX_AZ: continue
    ids.append(row.id); n_az += int(row.az)
    if len(ids) == 100: break

n_docked_picks = len(ids)
if len(ids) < 100:   # fill with undocked, non-azaindole ML picks from the pool
    rest = pd.read_csv("dock_pool.csv")
    ids += [i for i in rest.id if i not in set(ids)][: 100 - len(ids)]

open("submission_final.txt", "w").write("\n".join(ids) + "\n")
print("IDs:", len(ids), "| unique:", len(set(ids)), "| from docked:", n_docked_picks,
      "| azaindoles:", n_az)
files.download("submission_final.txt")

In [ ]:
# ===== 6. CHECK =====
ids = open("submission_final.txt").read().split()
print(len(ids), len(set(ids)), ids[:3])
import pandas as pd
dk = pd.read_csv("dock_results.csv")
print("docked:", len(dk), "| hinge contact:", int(dk.hinge.sum()))
print(dk.aff.describe().round(2))